In [1]:
import pandas as pd
from rapidfuzz import process
import sys
import os

# Go up one level from 'notebooks' to the project root
sys.path.append(os.path.abspath(".."))

In [2]:
df = pd.read_csv("../data/messy_inventory_data.csv")
df.head()

,SKU_ID,PRODUCT NAME,category_name,price,QTY,added_date,Supplier_Info
0,1621,Omega-3 Fish Oil,DRINKS,TBD,258.0,2024-11-06,"Carter, Kennedy and Rios"
1,4340,M4gn3sium Citr4t3,Supps,"59,24",0.0,2023-00-01,"Bennett, Bowen and Gamble"
2,3898,L-Glutamine_Powder,DRINKS,$46.04,NaN,NaN,"Hernandez, Garcia and Crawford"
3,2440,BCAA FLASH,DRINKS,NaN,NaN,2025-05-11,"Lane, Alvarado and Tran"
4,5658,Pre-Workout Explosion,DRINKS,$54.14,NaN,2024-06-01,Blair-Bond


In [3]:
df.shape

(5109, 7)

In [4]:
df.isna().sum()

  SKU_ID            0
PRODUCT NAME        0
category_name    1000
price             994
QTY              1641
added_date       1677
Supplier_Info    1034
dtype: int64

In [5]:
df.duplicated().sum()

np.int64(109)

In [6]:
clean_df = pd.read_csv("../data/clean_inventory.csv")
clean_df.head()

,sku_id,product_name,category_name,price,qty,added_date,supplier_info,category_name_was_missing,price_was_missing,qty_was_missing,added_date_was_missing,supplier_info_was_missing
0,1621,Omega-3 Fish Oil,Drinks,51.72,258.0,2024-11-06,"Carter, Kennedy and Rios",0,1,0,0,0
1,4340,M4Gn3Sium Citr4T3,Supps,59.24,0.0,2024-11-06,"Bennett, Bowen and Gamble",0,0,0,1,0
2,3898,L-Glutamine_Powder,Drinks,46.04,0.0,2024-11-06,"Hernandez, Garcia and Crawford",0,0,1,1,0
3,2440,Bcaa Flash,Drinks,51.72,0.0,2025-05-11,"Lane, Alvarado and Tran",0,1,1,0,0
4,5658,Pre-Workout Explosion,Drinks,54.14,0.0,2024-06-01,Blair-Bond,0,0,1,0,0


Some products appear under slightly different names, e.g.,

- L-Glutamine_Powder | L-Glut4Min3 Powd3R | L-Glutamine Powder !!!
- Omega-3_Fish_Oil | Omega-3 Fish Oil | Om3G4-3 Fish Oil 
- Vit4Min D3 Gold | Vitamin D3 Gold | Vitamin_D3_Gold  
- Vegan Pea Protein !!! | V3G4N P34 Prot3In 

This shows three distinct problems here:

- Format Noise: Underscores and "fat-finger" characters (like !!!).
- Leet Speak/Typos: Numbers replacing letters (like 3 for e, 4 for a).
- Fuzzy Variants: Subtle differences in spacing or naming that refer to the same item.

I solved this using Fuzzy Matching which allows the script to say: "This string looks 90% like 'Vegan Pea Protein', so I will rename it to that.

In [7]:
clean_df['category_name'].value_counts()

category_name
Drinks         1056
Nutr.          1016
Supplements     992
Unknown         977
Supps           959
Name: count, dtype: int64

In [8]:
clean_df['supplier_info'].value_counts()

supplier_info
Unknown                   1011
Johnson LLC                  9
Smith and Sons               9
Williams Inc                 8
Johnson Group                7
                          ... 
Reynolds LLC                 1
Morales-Tapia                1
Jones, Ward and Foster       1
Rodriguez-Jackson            1
Clark and Sons               1
Name: count, Length: 3663, dtype: int64

In [9]:
clean_df['sku_id'].nunique()

4762

## Data Cleaning Hierarchy

1. Clean Categories
2. Clean Product Names
3. Group Prices to Solve Variants of Same Product
4. Clean SKUs

- Cleaning names before the categories has a risk merging a "Vitamin D3 Drink" with a "Vitamin D3 Tablet," which are different products.
- By starting with the Category, I created "buckets." Then, within those buckets, I safely cleaned names and SKUs.
- I used price as a guide to cleaning SKUs:
    - Within 1 dollar difference as same product probably recorded at different price points including but not limited to promotions such as discounts;
    - A small difference to indicate variants of same product, e.g, chocolate and vanila flavours, and therefore must retain different SKUs;
    - A bigger difference to indicate different sizes of the product, e.g., 500g and 2kg, which were also given different SKUs.

In [10]:
d3 = clean_df[clean_df['product_name'] == "Vitamin D3 Gold"]
d3

,sku_id,product_name,category_name,price,qty,added_date,supplier_info,category_name_was_missing,price_was_missing,qty_was_missing,added_date_was_missing,supplier_info_was_missing
14,102,Vitamin D3 Gold,Supplements,95.97,411.0,2025-11-06,Jones Group,0,0,0,0,0
45,1487,Vitamin D3 Gold,Unknown,40.73,1.0,2024-04-29,Palmer-Garcia,1,0,0,0,0
82,4694,Vitamin D3 Gold,Nutr.,51.72,544.0,2025-05-13,Mitchell PLC,0,1,0,1,0
105,3834,Vitamin D3 Gold,Supplements,90.84,741.0,2026-01-27,"White, Bradshaw and Manning",0,0,0,1,0
124,1518,Vitamin D3 Gold,Drinks,76.75,0.0,2025-12-01,Brown-Henry,0,0,0,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...
4892,1249,Vitamin D3 Gold,Nutr.,51.72,0.0,2024-05-25,Stevens-Hurley,0,1,1,1,0
4900,5244,Vitamin D3 Gold,Drinks,25.03,0.0,2024-05-30,Mckenzie-Nunez,0,0,1,1,0
4912,3398,Vitamin D3 Gold,Unknown,51.72,280.0,2024-12-21,Anderson and Sons,1,1,0,1,0
4922,5247,Vitamin D3 Gold,Supplements,15.87,0.0,2026-03-12,Murray Group,0,0,1,0,0


In [11]:
d3['price'].value_counts()

price
51.72    89
53.33     2
55.69     2
95.97     1
40.73     1
         ..
60.00     1
49.50     1
25.03     1
15.87     1
85.18     1
Name: count, Length: 134, dtype: int64

In [12]:
all_d3 = clean_df[clean_df['product_name'].str.contains('D3')]
all_d3.shape

(339, 12)

In [13]:
all_d3['price'].value_counts().sort_values(ascending = False)

price
51.72    137
57.98      2
53.33      2
55.69      2
25.67      1
        ... 
81.46      1
93.72      1
44.95      1
94.32      1
36.51      1
Name: count, Length: 200, dtype: int64

In [14]:
all_d3['price'].head(20).sort_values(ascending=False)

14     95.97
105    90.84
182    88.72
268    88.22
303    86.38
124    76.75
324    76.51
31     76.05
258    69.05
270    54.60
262    53.22
316    51.72
174    51.72
137    51.72
127    51.72
82     51.72
328    51.72
45     40.73
52     31.27
173    14.84
Name: price, dtype: float64

In [15]:
all_d3['sku_id'].value_counts()

sku_id
102     5
103     3
101     3
5219    1
1487    1
       ..
3398    1
5247    1
4797    1
1732    1
2504    1
Name: count, Length: 331, dtype: int64

## Clean Product Categories

In [16]:
# clean categories
#category map
CATEGORY_MAP = {
    'supps': 'Supplements',
    'supplements': 'Supplements',
    'nutr.': 'Supplements',
    'drinks': 'Drinks',
    'drnks': 'Drinks',
    'unknown': 'Uncategorized'
}

#import the function to clean categories 
from deep_clean import standardize_categories

df2 = standardize_categories(clean_df, 'category_name', CATEGORY_MAP)

df2['category_name'].value_counts()

category_name
Supplements      2967
Drinks           1056
Uncategorized     977
Name: count, dtype: int64

## Fuzzy Match to Clean Product Names

In [17]:
# create a master product list
reference_list = [
    "Optimum Whey", "Hydrolyzed Iso", "Vegan Pea Protein", "Creatine Monohydrate",
    "BCAA Flash", "Pre-Workout Explosion", "ZMA Night", "Omega-3 Fish Oil",
    "Multivitamin Daily", "Casein Slow Release", "L-Glutamine Powder",
    "Mass Gainer 3000", "Keto Electrolytes", "Magnesium Citrate", "Vitamin D3 Gold"
]

from deep_clean import standardize_product_names
   
df3 = standardize_product_names(df2, 'product_name', reference_list=reference_list)

print('\n')
print(df3.head(20))



    sku_id           product_name  category_name  price    qty  added_date  \
0     1621       Omega-3 Fish Oil         Drinks  51.72  258.0  2024-11-06   
1     4340      Magnesium Citrate    Supplements  59.24    0.0  2024-11-06   
2     3898     L-Glutamine Powder         Drinks  46.04    0.0  2024-11-06   
3     2440             BCAA Flash         Drinks  51.72    0.0  2025-05-11   
4     5658  Pre-Workout Explosion         Drinks  54.14    0.0  2024-06-01   
5     4115   Creatine Monohydrate  Uncategorized  14.17  295.0  2024-06-01   
6      103  Pre-Workout Explosion    Supplements  43.89    0.0  2024-06-01   
7     2300      Keto Electrolytes    Supplements  51.72   85.0  2024-06-01   
8     2838         Hydrolyzed Iso    Supplements  51.72    0.0  2024-06-01   
9     5840    Casein Slow Release    Supplements  51.72    0.0  2024-06-01   
10    1704     Multivitamin Daily    Supplements   9.35    0.0  2025-07-21   
11    1173   Creatine Monohydrate  Uncategorized  51.72    0.0

In [18]:
# if we go back and recheck the D3 products to confirm cleaning
d3 = df3[df3['product_name'] == "Vitamin D3 Gold"]
d3

,sku_id,product_name,category_name,price,qty,added_date,supplier_info,category_name_was_missing,price_was_missing,qty_was_missing,added_date_was_missing,supplier_info_was_missing
14,102,Vitamin D3 Gold,Supplements,95.97,411.0,2025-11-06,Jones Group,0,0,0,0,0
31,5219,Vitamin D3 Gold,Supplements,76.05,27.0,2025-10-11,Williamson Inc,0,0,0,1,0
45,1487,Vitamin D3 Gold,Uncategorized,40.73,1.0,2024-04-29,Palmer-Garcia,1,0,0,0,0
52,4598,Vitamin D3 Gold,Uncategorized,31.27,0.0,2025-03-05,Simpson Group,1,0,1,1,0
82,4694,Vitamin D3 Gold,Supplements,51.72,544.0,2025-05-13,Mitchell PLC,0,1,0,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...
4928,101,Vitamin D3 Gold,Supplements,85.18,0.0,2025-05-22,Boone-Callahan,0,0,0,1,0
4957,4797,Vitamin D3 Gold,Supplements,6.89,0.0,2024-11-23,"Serrano, Clark and Long",0,0,1,0,0
4989,103,Vitamin D3 Gold,Supplements,57.98,0.0,2024-05-21,Hernandez and Sons,0,0,0,1,0
4992,1732,Vitamin D3 Gold,Supplements,51.72,0.0,2024-12-08,"Wilson, White and Craig",0,1,1,1,0


- There are now 339 products named **Vitamin D3 Gold**, confirming the product names cleaning step was successful

## Clean SKUs

In [19]:
# clean SKUs
group_cols = ['product_name', 'category_name']


from deep_clean import consolidate_skus

df4 = consolidate_skus(df3, group_cols= group_cols, price_col= 'price', tolerance_usd=1.00)
df4[['sku_id', 'product_name', 'category_name', 'price', 'consolidated_sku_id']].head()

,sku_id,product_name,category_name,price,consolidated_sku_id
144,3927,BCAA Flash,Drinks,12.36,BF-3927-0
148,4906,BCAA Flash,Drinks,16.09,BF-3927-1
730,2413,BCAA Flash,Drinks,19.41,BF-3927-2
969,5984,BCAA Flash,Drinks,21.90,BF-3927-3
3567,5817,BCAA Flash,Drinks,22.66,BF-3927-3


In [20]:
d3 = df4[df4['product_name'] == "Vitamin D3 Gold"]
d3[['sku_id', 'product_name', 'category_name', 'price', 'consolidated_sku_id']]

,sku_id,product_name,category_name,price,consolidated_sku_id
2734,5436,Vitamin D3 Gold,Drinks,5.29,VD-5436-0
3656,3738,Vitamin D3 Gold,Drinks,5.73,VD-5436-0
3200,3683,Vitamin D3 Gold,Drinks,5.85,VD-5436-0
1776,2205,Vitamin D3 Gold,Drinks,9.18,VD-5436-1
685,2470,Vitamin D3 Gold,Drinks,11.26,VD-5436-2
...,...,...,...,...,...
1624,2237,Vitamin D3 Gold,Uncategorized,93.72,VD-5372-28
1643,2837,Vitamin D3 Gold,Uncategorized,94.32,VD-5372-28
1821,2788,Vitamin D3 Gold,Uncategorized,96.67,VD-5372-29
2451,4889,Vitamin D3 Gold,Uncategorized,97.59,VD-5372-29


In [21]:
d3['consolidated_sku_id'].nunique()

85

- After consolidating SKUs, now the first three D3 products, which are all drinks with original sku_ids (5436, 3738, and	3683) all have been given the same new SKU, **VD-5436-0**

- The D3 product list (SKUs) have also reduced from 339 to 85 unique products. 

### Recheck the cleaned inventory after running the scripts through teminal

In [22]:
df = pd.read_csv("../data/final_clean_inventory.csv")
df.head()

,sku_id,product_name,category_name,price,qty,added_date,supplier_info,category_name_was_missing,price_was_missing,qty_was_missing,added_date_was_missing,supplier_info_was_missing,price_gap,cluster_id,consolidated_sku_id
0,3927,BCAA Flash,Drinks,12.36,0.0,2025-10-23,Unknown,0,0,1,1,1,0.00,0,BF-3927-0
1,4906,BCAA Flash,Drinks,16.09,533.0,2025-06-07,Reyes-Ramirez,0,0,0,1,0,3.73,1,BF-3927-1
2,2413,BCAA Flash,Drinks,19.41,0.0,2024-12-24,Gordon-Powell,0,0,0,0,0,3.32,2,BF-3927-2
3,5984,BCAA Flash,Drinks,21.90,0.0,2024-10-20,Gates Group,0,0,0,0,0,2.49,3,BF-3927-3
4,5817,BCAA Flash,Drinks,22.66,0.0,2024-04-26,Valencia Group,0,0,0,1,0,0.76,3,BF-3927-3


In [23]:
d3 = df[df['product_name'] == "Vitamin D3 Gold"]
d3[['sku_id', 'product_name', 'category_name', 'price', 'consolidated_sku_id']]

,sku_id,product_name,category_name,price,consolidated_sku_id
4358,5436,Vitamin D3 Gold,Drinks,5.29,VD-5436-0
4359,3738,Vitamin D3 Gold,Drinks,5.73,VD-5436-0
4360,3683,Vitamin D3 Gold,Drinks,5.85,VD-5436-0
4361,2205,Vitamin D3 Gold,Drinks,9.18,VD-5436-1
4362,2470,Vitamin D3 Gold,Drinks,11.26,VD-5436-2
...,...,...,...,...,...
4692,2237,Vitamin D3 Gold,Uncategorized,93.72,VD-5372-28
4693,2837,Vitamin D3 Gold,Uncategorized,94.32,VD-5372-28
4694,2788,Vitamin D3 Gold,Uncategorized,96.67,VD-5372-29
4695,4889,Vitamin D3 Gold,Uncategorized,97.59,VD-5372-29
